<a href="https://colab.research.google.com/github/ibmm-unibe-ch/FrankenMSA/blob/ngrok/app/FrankenMSA_app_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://github.com/ibmm-unibe-ch/FrankenMSA/blob/dev/app/assets/frankenmsa_header.png?raw=true)

# FrankenMSA
This notebook launches the [FrankenMSA App](https://github.com/ibmm-unibe-ch/FrankenMSA/tree/main/) **in Google Colab** to provide a GUI for manipulating Multiple Sequence Alignments (MSAs).

The current version runs ProteinMPNN **Colab-only via ngrok**. A public ngrok URL will be created to access the app.

How to use:
1. Run the cell below and enter your **ngrok authtoken** when prompted (get one from [ngrok](https://ngrok.com/), a free account is enough).
2. Run all cells; once you see the printed **ngrok tunnel** URL, click it to open the app in a new tab.
3. If the link expires or an error occurs, just re-run the “launch” cell.

Tip: use “Runtime” → “Run all” (or `Ctrl + F9`) to execute all cells.

# Code

In [ ]:
# === Cell 1: Hard reset + fresh clone of the RIGHT branch ===
import os, sys, shutil, subprocess, time, json, textwrap, pathlib

# --- Clean any old tunnels/processes (best-effort) ---
try:
    from pyngrok import ngrok
    for t in ngrok.get_tunnels():
        try:
            ngrok.disconnect(t.public_url)
        except Exception:
            pass
    ngrok.kill()
except Exception:
    pass

# kill old app processes if any
!pkill -f "python app/app.py" 2>/dev/null || true
!pkill -f "gunicorn"          2>/dev/null || true

# --- Fresh clone the correct branch ---
shutil.rmtree("/content/FrankenMSA", ignore_errors=True)
!git clone -q --single-branch -b feature/colab-runner https://github.com/ibmm-unibe-ch/FrankenMSA.git /content/FrankenMSA

# show branch & last commit for sanity
!git -C /content/FrankenMSA rev-parse --abbrev-ref HEAD
!git -C /content/FrankenMSA log -1 --pretty=oneline

# --- Base deps for the web app / ngrok  ---
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "dash==2.16.1", "dash-bootstrap-components==1.5.0",
                           "plotly==5.24.1", "dash-bio==1.0.2", "pyngrok"])
    print("✅ base deps installed")
except Exception as e:
    print("⚠️ deps warning:", e)

# --- Inject build banner into inversefold.py ---
try:
    commit = subprocess.check_output(
        ["git","-C","/content/FrankenMSA","rev-parse","--short","HEAD"],
        text=True
    ).strip()
    p = pathlib.Path("/content/FrankenMSA/app/pages/inversefold.py")
    s = p.read_text(encoding="utf-8")
    if "INVERSEFOLD_BUILD_ID" not in s:
        s = s.replace(
            "def layout():",
            f"INVERSEFOLD_BUILD_ID = 'IF-build:{commit}'\n\ndef layout():"
        ).replace(
            'html.H1("Inverse Fold with ProteinMPNN")',
            'html.H1("Inverse Fold with ProteinMPNN"), html.Small(INVERSEFOLD_BUILD_ID, style={"marginLeft":"8px","opacity":0.6})'
        )
        p.write_text(s, encoding="utf-8")
        print("🧩 injected build banner:", commit)
    else:
        print("ℹ️ build banner already present")
except Exception as e:
    print("⚠️ build banner injection skipped:", e)

# --- Export /content/colab_bridge.py  ---
bridge_code = r'''
import sys

if "/content/FrankenMSA/app" not in sys.path:
    sys.path.append("/content/FrankenMSA/app")

import proteinmpnn_runner as pmr

def parse_params(qs_or_url: str):
    """
    Normalize FrankenMSA URL/query-string to parameter dict.
    (Web app expects this symbol name.)
    """
    return pmr.parse_param_string(qs_or_url)

def run_proteinmpnn(
    sampling_temp=1.0, num_seqs=128, pdb_code="", design_csv="", fixed_csv="",
    homomer=True, model_name="v_48_020", use_soluble_model=False, ca_only=False,
    clean_workspace=True, allow_upload=False, auto_download=False,
    pdb_path=""  # optional: if later you want to pass an uploaded local PDB file path
):
    
    return pmr.run_proteinmpnn(
        sampling_temp=sampling_temp,
        num_seqs=num_seqs,
        pdb_code=pdb_code,
        design_csv=design_csv,
        fixed_csv=fixed_csv,
        homomer=homomer,
        model_name=model_name,
        use_soluble_model=use_soluble_model,
        ca_only=ca_only,
        allow_upload=allow_upload,
        clean_workspace=clean_workspace,
        auto_download=auto_download,
        pdb_path=pdb_path,
    )
'''
with open("/content/colab_bridge.py", "w") as f:
    f.write(bridge_code)


if "/content" not in sys.path:
    sys.path.append("/content")
!mkdir -p /content/ProteinMPNN/uploads

# Smoke test
import importlib
colab_bridge = importlib.import_module("colab_bridge")
print("✅ colab_bridge exported:", [n for n in dir(colab_bridge) if not n.startswith("_")])

print("✅ Cell 1 done. Go to Cell 2.")

In [ ]:
# === Cell 2: Start app via ngrok, reusing or creating tunnel ===
# --- Kill old FrankenMSA processes ---
import os, signal, subprocess, time


!pkill -f "app/app.py" 2>/dev/null || true
!pkill -f "gunicorn" 2>/dev/null || true
!pkill -f "ngrok" 2>/dev/null || true

time.sleep(1)
print("🧹 Cleaned up old FrankenMSA + ngrok processes.")

import os, sys, time, re
from pathlib import Path
from pyngrok import ngrok, conf

ROOT = "/content/FrankenMSA"
assert os.path.isdir(ROOT), "FrankenMSA repo not found. Run Cell 1 first."


import getpass
PORT = 8050
token = getpass.getpass("Enter ngrok authtoken (hidden): ").strip().strip("'").strip('"')
conf.get_default().auth_token = token

public_url = None
try:
    
    for t in ngrok.get_tunnels():
        addr = (t.config or {}).get("addr","")
        if addr.endswith(f":{PORT}"):
            public_url = t.public_url
            print("♻️ Reusing existing tunnel:", public_url)
            break
    
    if not public_url:
        tun = ngrok.connect(addr=f"0.0.0.0:{PORT}", proto="http")
        public_url = tun.public_url
        print("✅ Created new tunnel:", public_url)
except Exception as e:
    msg = str(e)
    m = re.search(r"https?://[a-z0-9\-]+\.ngrok-[\w\-]+\.(?:dev|app)", msg)
    if m:
        public_url = m.group(0)
        print("♻️ Using tunnel from error message:", public_url)
    else:
        raise


patch_total = 0
for p in Path(f"{ROOT}/app").rglob("*.py"):
    s = p.read_text(encoding="utf-8", errors="ignore")
    s2, n1 = re.subn(r",\s*\{\s*['\"]prevent_initial_call['\"]\s*:\s*True\s*\}", ", prevent_initial_call=True", s)
    s3, n2 = re.subn(r"clientside_callback\((.*?)\s*,\s*\{\s*['\"]prevent_initial_call['\"]\s*:\s*True\s*\}\s*\)",
                     r"clientside_callback(\1, prevent_initial_call=True)", s2, flags=re.DOTALL)
    if n1 or n2:
        p.write_text(s3, encoding="utf-8")
        patch_total += n1 + n2
print(f"🩹 Patched {patch_total} place(s)")


import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ROOT])
env = os.environ.copy()
env["PORT"], env["HOST"] = str(PORT), "0.0.0.0"
env["PYTHONPATH"] = "/content:" + env.get("PYTHONPATH", "")

proc = subprocess.Popen(
    [sys.executable, "app/app.py"],
    cwd=ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)


start = time.time()
lines = []
while time.time() - start < 25:
    ln = proc.stdout.readline()
    if ln:
        lines.append(ln.rstrip())
        if "Running on" in ln or "Dash is running" in ln:
            break
    else:
        time.sleep(0.2)

print("\n---- recent logs ----")
print("\n".join(lines[-20:]))
print("---------------------")
print("🌐 Open:", public_url)
print("✅ Look for the banner 'IF-build:xxxx' on the page header to confirm version")
print("📡 Tailing FrankenMSA app logs (Ctrl+C to stop):")
while True:
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    print(line, end="")